# Multi-Model AI Assistant (Groq + Gemini + OpenRouter)

### BUSINESS CHALLENGE:

Build one chat assistant that can run on **three different free-tier providers** -
Groq, Gemini, and OpenRouter - letting the customer switch models
mid-conversation and compare answers. On top of that, the assistant can look up **real
hotel prices** via the Makcorps API (same tool-calling pattern as Day 3, but hitting a
real API instead of SQLite), and it **speaks its replies out loud** using free
text-to-speech.

This notebook covers:
- Reusing the same tool-calling loop from Day 3, but making it work across **multiple
  providers** by swapping which client/model we send it to
- Calling a **real third-party API** (Makcorps) as a tool, not just a local database
- Turning text replies into **speech** using `gTTS` (free, no API key needed)
- A Gradio UI with a provider dropdown, a chat window, and an audio player for the
  spoken reply


In [22]:
import os
import json
import requests
from datetime import date, timedelta

from dotenv import load_dotenv
from openai import OpenAI
from gtts import gTTS
import gradio as gr

#### Step 1: Load API keys

Reads all four keys from `.env` - three LLM provider keys plus the Makcorps key for
hotel prices.

In [2]:
load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
makcorps_api_key = os.getenv("MAKCORPS_API_KEY")

for name, key in [
    ("Groq", groq_api_key),
    ("Google", google_api_key),
    ("OpenRouter", openrouter_api_key),
    ("Makcorps", makcorps_api_key),
]:
    if key:
        print(f"{name} API Key exists and begins {key[:8]}")
    else:
        print(f"{name} API Key not set - check your .env file")

Groq API Key exists and begins gsk_ePDh
Google API Key exists and begins AQ.Ab8RN
OpenRouter API Key exists and begins sk-or-v1
Makcorps API Key exists and begins 6a6f74b8


#### Step 2: Set up the three provider clients

Same trick as Week 2 Day 1 - Gemini and OpenRouter both expose OpenAI-compatible
endpoints, so I reuse the same `OpenAI` client class for all three providers and just
swap `base_url` and the API key.

I also map each provider name to the model I'll use for it, so the rest of the
notebook can just say "use groq" or "use gemini" without repeating model names everywhere.

In [3]:
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
openrouter = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

providers = {
    "Groq (llama-3.3-70b)": {"client": groq, "model": "llama-3.3-70b-versatile"},
    "Gemini (flash)": {"client": gemini, "model": "gemini-flash-latest"},
    "OpenRouter (glm-4.6v)": {"client": openrouter, "model": "z-ai/glm-4.6v"},
}

#### Step 3: The hotel price tool (real API call)

Same idea as Day 3's `search_hotels`, but instead of querying SQLite, this calls
Makcorps' free hotel price endpoint for a real city. It returns real hotel names and
prices from sites like Booking.com and Expedia.

Note: the free endpoint only supports a city name (no check-in/check-out dates, no
guest count) and returns up to 30 hotels - I trim that down to the 5 cheapest so the
model isn't overwhelmed with data.

In [ ]:

def get_hotel_id(hotel_name: str) -> str | None:
    url = "https://api.makcorps.com/mapping"
    params = {"api_key": makcorps_api_key, "name": hotel_name}

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    results = response.json()

    for result in results:
        if result.get("type") == "HOTEL":
            return str(result["document_id"])
    return None


def get_hotel_prices(hotel_name: str) -> str:
    hotel_id = get_hotel_id(hotel_name)
    if not hotel_id:
        return f"Could not find a hotel matching '{hotel_name}'."

    checkin = (date.today() + timedelta(days=30)).isoformat()
    checkout = (date.today() + timedelta(days=31)).isoformat()

    url = "https://api.makcorps.com/hotel"
    params = {
        "api_key": makcorps_api_key,
        "hotelid": hotel_id,
        "rooms": 1,
        "adults": 1,
        "checkin": checkin,
        "checkout": checkout,
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    vendors = data.get("comparison", [[]])[0]
    results = []
    for entry in vendors:
        for key, value in entry.items():
            if key.startswith("vendor"):
                idx = key.replace("vendor", "")
                price = entry.get(f"price{idx}")
                if price:
                    results.append(f"{value}: {price}")

    if not results:
        return f"No prices found for {hotel_name}."

    return f"Prices for {hotel_name} (for {checkin} to {checkout}):\n" + "\n".join(results)

#### Step 4: Text-to-speech with gTTS

`gTTS` converts text to an MP3 file - no API key needed, free to use. I save each
reply to a temp file and hand that file path to Gradio's audio player.

In [13]:
def text_to_speech(text: str) -> str:
    output_path = "reply.mp3"
    tts = gTTS(text=text, lang="en")
    tts.save(output_path)
    return output_path

#### Step 5: Describe the tool to the LLM

Same schema pattern as Day 3, just one tool this time.

In [23]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_hotel_prices",
            "description": "Get real prices for a specific named hotel. Requires a hotel name, not just a city - if the customer only gives a city, ask them to name a specific hotel (or suggest a well-known one in that city) before calling this.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel_name": {"type": "string", "description": "Name of the hotel, e.g. 'Pod 51 Hotel New York'"},
                },
                "required": ["hotel_name"],
            },
        },
    },
]

available_functions = {"get_hotel_prices": get_hotel_prices}

system_prompt = """
You are a helpful travel assistant. Keep replies short - 1 to 3 sentences.
Use the get_hotel_prices tool whenever a customer asks about prices for a specific hotel.
This tool needs a hotel NAME, not just a city - if the customer only mentions a city,
ask them to name a specific hotel before searching.
Never invent hotel names or prices - only state what the tool returns.
"""

#### Step 6: The multi-provider tool-calling function

This is the same tool-calling loop as Day 3, but it now takes `client` and `model` as
arguments instead of being hardcoded to Groq - so the exact same function works no
matter which provider the customer picks from the dropdown.

In [24]:
def get_reply(client, model, message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    while True:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.3,
        )
        reply = response.choices[0].message

        if not reply.tool_calls:
            return reply.content

        messages.append(reply)

        for tool_call in reply.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            function = available_functions[function_name]

            result = function(**function_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

#### Step 7: The Gradio UI

I use `gr.Blocks` instead of `gr.ChatInterface` here, because I need an extra piece
of UI - the provider dropdown - alongside the chat and the audio player, and Blocks
gives the full control over that layout.

How it works:
1. Customer types a message and picks a provider from the dropdown
2. `respond()` cleans the chat history down to `role`/`content` (same fix as Week 2/3),
   sends it to the chosen provider via `get_reply()`, and gets back a text answer
3. That answer is converted to speech with `text_to_speech()`
4. The chat window updates with the text, and the audio player updates with the voice

In [25]:
def respond(message, chat_history, provider_name):
    cleaned_history = [{"role": h["role"], "content": h["content"]} for h in chat_history]

    provider = providers[provider_name]
    reply_text = get_reply(provider["client"], provider["model"], message, cleaned_history)

    audio_path = text_to_speech(reply_text)

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": reply_text})

    return "", chat_history, audio_path


with gr.Blocks(title="Multi-Model Travel Assistant") as demo:
    gr.Markdown("# Multi-Model Travel Assistant")

    provider_dropdown = gr.Dropdown(
        choices=list(providers.keys()),
        value=list(providers.keys())[0],
        label="Choose a provider",
    )

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Your message", placeholder="e.g. What are hotel prices in Mumbai?")
    audio_output = gr.Audio(label="Spoken reply", autoplay=True)

    msg.submit(
        respond,
        inputs=[msg, chatbot, provider_dropdown],
        outputs=[msg, chatbot, audio_output],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
